In [0]:

from environment import SILVER_TABLES,BRONZE_TABLES

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS chilean_news.checkpoints ")
spark.sql("CREATE SCHEMA IF NOT EXISTS chilean_news.golden_tables ")
spark.sql("CREATE VOLUME IF NOT EXISTS chilean_news.checkpoints.golden ")


In [0]:
import pyspark.sql.functions as F

In [0]:
def create_all_news():
    biobio_df = spark.read.table(SILVER_TABLES["news_biobio"]).withColumn("medio",F.lit("BioBioChile"))
    diario_fin_df = spark.read.table(SILVER_TABLES["news_diario_financiero"]).withColumn("medio",F.lit("Diario Financiero"))
    la_nacion_df = spark.read.table(SILVER_TABLES["news_la_nacion"]).withColumn("medio",F.lit("La Nación")) 
    the_clinic_df = spark.read.table(SILVER_TABLES["news_the_clinic"]).withColumn("medio",F.lit("The Clinic")) 

    return (biobio_df
            .union(diario_fin_df)
            .union(la_nacion_df)
            .union(the_clinic_df)
            )
    
all_news = create_all_news()

all_news.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("chilean_news.golden_tables.all_news")

In [0]:
from pyspark.sql import Window
an_tb = "chilean_news.golden_tables.all_news"
windowSpec = Window.partitionBy("medio").orderBy("fecha_publicacion").rowsBetween(-1,1)

def add_sentiment(table):
    df = spark.read.table(table).dropDuplicates(["link"])
    
    return (df.withColumn("sentiment",F.expr("ai_analyze_sentiment(texto_completo)"))
            .where(F.col("sentiment").isNotNull())
            .withColumn("sentiment_grade",
                F.when(F.expr("sentiment = 'positive'"),1)
                .when(F.expr("sentiment = 'negative'"),-1)
                .when(F.expr("sentiment = 'neutral'"),0)
                .when(F.expr("sentiment = 'mixed'"),0)
                .otherwise(0)
            )
            .withColumn("sentiment_mobile_avg",F.avg("sentiment_grade").over(windowSpec))
    )

sentiment_table = add_sentiment(an_tb)

sentiment_table.write.mode("overwrite").saveAsTable(an_tb + "analyzed")